In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'c:\\Users\\MATT\\Documents\\End-to-end-chicken-Disease-Classification'

In [3]:
import tensorflow as tf


In [4]:
model = tf.keras.models.load_model("artifacts/training/trained_model_best.keras")

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    label_csv_path: Path 
    image_data_dir: Path       
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [6]:
import sys
import os

# Add src directory to Python path
src_path = os.path.join(os.getcwd(), 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


from Chicken_Disease_Classification.constant import *
from Chicken_Disease_Classification.utils.common import read_yaml, create_directories,save_json


In [7]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)
tf.compat.v1.enable_eager_execution()
import time 
import os
from pathlib import Path


[2025-10-06 12:04:40,268: WARNING: module_wrapper: From C:\Users\MATT\AppData\Local\Temp\ipykernel_24240\3091630949.py:3: The name tf.enable_eager_execution is deprecated. Please use tf.compat.v1.enable_eager_execution instead.
]


In [ ]:

from Chicken_Disease_Classification.config import *
class ConfigurationManager:
    def __init__(self, 
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path("artifacts/training/trained_model_best.keras"),
            label_csv_path=Path(self.config.training.label_csv_path),       
            image_data_dir=Path(self.config.training.image_data_dir),       
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        
        return eval_config

In [9]:
from urllib.parse import urlparse 

In [10]:
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
import numpy as np
import os

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.model = tf.keras.models.load_model(self.config.path_of_model)
        self.valid_generator = None
        self.accuracy = None
        self.loss = None

    def _valid_generator(self):
        df = pd.read_csv(self.config.label_csv_path)

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_dataframe(
            dataframe=df,
            directory=self.config.image_data_dir,
            x_col='images',
            y_col='label',
            subset="validation",
            shuffle=False,
            class_mode='categorical',
            **dataflow_kwargs
        )

        print(f"Validation generator created with {self.valid_generator.samples} samples")

    def run_evaluation(self):
        if self.valid_generator is None:
            self._valid_generator()

        # Get loss and accuracy from Keras
        self.loss, keras_accuracy = self.model.evaluate(self.valid_generator, verbose=0)

        # Manual accuracy (optional, for comparison)
        predictions = self.model.predict(self.valid_generator)
        predicted_labels = np.argmax(predictions, axis=1)
        true_labels = self.valid_generator.classes
        self.accuracy = accuracy_score(true_labels, predicted_labels)

        print(f"Evaluation Accuracy (manual): {self.accuracy:.4f}")
        print(f"Evaluation Loss: {self.loss:.4f}")

    def save_score(self):
        score_dir = "artifacts/evaluation"
        os.makedirs(score_dir, exist_ok=True)

        score_path = os.path.join(score_dir, "score.txt")
        with open(score_path, "w") as f:
            f.write(f"Loss: {self.loss:.4f}\n")
            f.write(f"Accuracy: {self.accuracy:.4f}\n")

        print(f"Score saved to {score_path}")

In [11]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.run_evaluation()
    evaluation.save_score()

except Exception as e:
    raise e

[2025-10-06 12:04:40,435: INFO: common: Directory created at: artifacts]
Found 1613 validated image filenames belonging to 4 classes.
Validation generator created with 1613 samples


c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
c:\Users\MATT\Documents\End-to-end-chicken-Disease-Classification\venv\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


101/101 ━━━━━━━━━━━━━━━━━━━━ 305s 3s/step
Evaluation Accuracy (manual): 0.8617
Evaluation Loss: 0.4205
Score saved to artifacts/evaluation\score.txt
